# ctxblame on Google Colab (Zero-API-Cost via vLLM)

Run `ctxblame` experiments at **$0 API cost** on a Colab **L4 (24 GB)** or **A100 (40 GB)** GPU with **vLLM automatic prefix caching** enabled.

What this notebook does:
1. Clones/installs `ctxblame` and `vllm`.
2. Launches a local OpenAI-compatible `vllm serve` on `http://127.0.0.1:8000/v1` with `--enable-prefix-caching` and tool-call parsing.
3. Either **replays existing OpenAI-format traces** (`--base-url`, `--model`, `--free`) OR **records a brand-new coding agent trace** through `ctxblame record` and runs conditional `ctxblame why` on it.

In [ ]:
!nvidia-smi
# Optional: mount Google Drive to persist traces and reports across Colab sessions
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Install vLLM and ctxblame (replace REPO_URL with your fork if needed)
REPO_URL = "https://github.com/kekeandzeyu/ctxblame.git"
!pip install -q vllm
!rm -rf /content/ctxblame && git clone {REPO_URL} /content/ctxblame
!pip install -q -e /content/ctxblame

In [ ]:
# Start vLLM in the background with automatic prefix caching + tool calling.
# Recommended models:
#   - L4 (24 GB):  Qwen/Qwen3-8B   (or Qwen/Qwen2.5-Coder-14B-Instruct-AWQ)
#   - A100 (40GB): Qwen/Qwen3-14B  (or Qwen/Qwen3-32B-AWQ)
import subprocess, time, urllib.request

MODEL = "Qwen/Qwen3-8B"
TOOL_PARSER = "hermes"  # Qwen3/Qwen2.5 tool call parser in vLLM

vllm_log = open("/tmp/vllm.log", "w")
vllm_proc = subprocess.Popen([
    "vllm", "serve", MODEL,
    "--host", "127.0.0.1", "--port", "8000",
    "--enable-prefix-caching",
    "--enable-auto-tool-choice",
    "--tool-call-parser", TOOL_PARSER,
    "--max-model-len", "24576",
    "--gpu-memory-utilization", "0.90",
], stdout=vllm_log, stderr=subprocess.STDOUT)

for _ in range(180):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2) as r:
            if r.status == 200:
                print("vLLM is ready!", r.read().decode()[:200])
                break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("vLLM failed to start; inspect /tmp/vllm.log")

In [ ]:
# 1) Quick zero-cost check: run ctxblame why on an existing Phase 1 OpenAI trace using the Colab model
%cd /content/ctxblame
!ctxblame why experiments/phase1/results/traces/openai-1.jsonl \
    --step 5 \
    --key-arg 'command~^python -m pytest' \
    --base-url http://127.0.0.1:8000/v1 \
    --model {MODEL} \
    --free \
    --concurrency 8

In [ ]:
# 2) Record a brand-new multi-turn agent trace against vLLM through ctxblame record
import subprocess, time

TRACE = "/content/ctxblame/traces/colab-qwen3.jsonl"
proxy_proc = subprocess.Popen([
    "ctxblame", "record",
    "--upstream", "http://127.0.0.1:8000/v1",
    "--port", "8791",
    "--trace", TRACE,
])
time.sleep(2)

!python experiments/phase1/agent.py \
    --format openai \
    --base-url http://127.0.0.1:8791 \
    --model {MODEL} \
    --no-thinking \
    --max-tokens 4096 \
    --workspace /tmp/ws

proxy_proc.terminate()
!ctxblame steps {TRACE}

In [ ]:
# 3) Inspect vLLM prefix-cache hit metrics after running ctxblame
!curl -s http://127.0.0.1:8000/metrics | grep -E "prefix_cache|prompt_tokens|generation_tokens" | head -n 20